# Self-Supervised Learning with PyTorch: An Introduction

In this notebook, we will explore the concept of learning useful representations from data without relying on extensive labeled datasets. Self-supervised learning allows us to train models on a "pretext task" using unlabeled data, enabling the model to extract features that can be reused for downstream tasks.

**What will we do?**
1. Train a feature extractor on a pretext task: predicting the rotation of images from the CIFAR-10 dataset.
2. Use the learned features to train a linear classifier on a small labeled subset of the dataset.
3. Compare the performance of this approach with training a feature extractor and classifier from scratch using the same small labeled subset.

**Why is this important?**  

The pretext task helps the model learn general and transferable features that reduce the need for labeled examples in the downstream task. This is particularly valuable when labeled data is scarce or expensive to obtain.

**The tutorial is based on page 11 of the lecture notes**

In [7]:
# Importing libraries
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import resnet18
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Setting the device for the models

In [8]:
# Define transformations for CIFAR10
def get_transforms():
    return transforms.Compose([
        transforms.RandomRotation(90), # Should probably remove this, please experiment, will do it later locally in the traning loop
        transforms.ToTensor(), # Convert the image to a tensor with pixels in the range [0, 1]
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]), # Normalize the image with mean and standard deviation
    ])

In [11]:
# Define the CNN feature extractor 
# This corresponds to page 11 of the lecture notes, the purple feature extractor
# If your computer cannot handle the ResNet18 model, you can use a simpler CNN model of your choice 
class FeatureExtractor(nn.Module):
    def __init__(self):
        super(FeatureExtractor, self).__init__()
        self.feature_extractor = resnet18(pretrained=False, num_classes=4) # it is impotant to set pretrained to False, otherwise the model will download the weights of the pre-trained model
        self.feature_extractor.fc = nn.Identity()  # Remove fully connected layer by setting it to Identity

    def forward(self, x):
        return self.feature_extractor(x)

In [1]:
# Test the feature extractor dont make a def
feature_extractor = FeatureExtractor()
x = torch.randn(4, 3, 32, 32) # shape (batch_size, channels, height, width), needs batchszie more than 1 for resnet18
y = feature_extractor(x)
print(y.shape) # Expected output: torch.Size([4, 512]) corrresponding to (batch_size, number of features)

In [15]:
# Define the full model with a small head
# This corresponds to page 11 of the lecture notes, the full model on the left side
# Your model take in an image and output the rotation class
# You must combine the feature extractor and the head
# The head is a simple linear layer with 4 output units
# Remember, later you will throw away the head and only use the feature extractor
class RotationClassifier(nn.Module):
    def __init__(self):
        super(RotationClassifier, self).__init__()

        # Your code here

    def forward(self, x):

        # Your code here

        return x

In [ ]:
# Test the full model
model = RotationClassifier()
x = torch.randn(4, 3, 32, 32) # fake image, shape (batch_size, channels, height, width)
y = model(x)
print(y.shape) # Expected output: torch.Size([4, 4]) corresponding to (batch_size, number of rotations)

In [17]:
# Pretext task: Train on image rotations
def train_pretext_task():

    # Load CIFAR10 dataset
    dataset = datasets.CIFAR10(root="./data", train=True, transform=get_transforms(), download=True)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

    # Model and loss
    model = RotationClassifier().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Training loop
    for epoch in range(10):  # Keep epochs low for quick training
        model.train()
        for images, _ in dataloader:
            batch_size = images.size(0)

            # Create rotated images (This is all vectorized and makes four copies of all the images in the batch)
            rotated_images, labels = [], []
            for i in range(4):  # 4 rotations (0, 90, 180, 270 degrees)

                # Your code here, use torch.rot90 and torch.cat

            # Forward pass
            outputs = model(rotated_images.to(device))
            loss = criterion(outputs, labels)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
    return model.features

In [19]:
# Define a simple linear classifier for the downstream task
# This corresponds to page 11 of the lecture notes, the blue linear classifier on the right side
class LinearClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(LinearClassifier, self).__init__()

        # Your code here

In [ ]:
# Test the linear classifier
classifier = LinearClassifier(512, 10)
x = torch.randn(4, 512) # shape (batch_size, number of features) The features are the representation of the image
y = classifier(x)
print(y.shape) # Expected output: torch.Size([4, 10]) corresponding to (batch_size, number of classes in the downstream CIFAR10 task)

In [ ]:
# Downstream task: Function to train a linear classifier on CIFAR10
# We are going to take in the feature extractor from the pretext task and train a linear classifier on top of it
# This corresponds to page 11 of the lecture notes, the right side of the full model
# Note that the model will be trained on a small subset of CIFAR10
# The point of this task is to show that even with a small labeled dataset, we can achieve good performance
# by leveraging the pretext task over the entire unlabeled dataset to warm up the models weights
def train_linear_classifier(feature_extractor, train_loader, val_loader):
    model = LinearClassifier(input_dim=512, num_classes=10).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    feature_extractor.eval()  # Freeze feature extractor (we are not training it)

    for epoch in range(10):  # Small epochs for efficiency, adjust as needed
        model.train()
        for images, labels in train_loader:

            # Your code here
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Validation
    correct = 0
    total = 0
    model.eval()
    with torch.no_grad():
        for images, labels in val_loader:
            features = feature_extractor(images.to(device))
            outputs = model(features)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels.to(device)).sum().item()
    accuracy = 100 * correct / total
    return accuracy

After defining all the functions, you can run this block to evaluate weather the pretext task helps in the downstream task

In [ ]:
# Load CIFAR10 with labeled subset (standard normalization)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

# Load CIFAR10 dataset, 10 classes and 6000 images per class
train_dataset = datasets.CIFAR10(root="./data", train=True, transform=transform, download=True)
test_dataset = datasets.CIFAR10(root="./data", train=False, transform=transform, download=True)

# Create a small labeled subset of CIFAR10
# This corresponds to page 11 of the lecture notes, the small labeled set in blue on the right side
small_labeled_set = Subset(train_dataset, np.random.choice(len(train_dataset), 500, replace=False))
train_loader = DataLoader(small_labeled_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False) # note that the test_loader is the entire test set

In [ ]:
# Pretext task, training the feature extractor (Left side of page 11)
feature_extractor = train_pretext_task()

# Linear evaluation on representation learned by pretext task (right side of page 11)
# Uses the pretext trained feature extractor to train a linear classifier on the small labeled subset (training set)
# Model is then evaluated on the test set (unseen large labeled set)
pretext_accuracy = train_linear_classifier(feature_extractor, train_loader, test_loader)
print(f"Accuracy with pretext task: {pretext_accuracy:.2f}%")

# Training from scratch to compare with pretext task
# We are training the feature extractor from scratch, i.e., not using the pretext task over the entire unlabeled dataset
# This simply trains the combined feature extractor and linear classifier on the small labeled subset and then evaluates on the test set
scratch_feature_extractor = FeatureExtractor().to(device) # reinitialize the feature extractor from scratch
scratch_accuracy = train_linear_classifier(scratch_feature_extractor, train_loader, test_loader) # train the linear classifier on the small labeled subset
print(f"Accuracy from scratch: {scratch_accuracy:.2f}%")

In [ ]:
# Discuss your results here 